# Assignment 6: Boston House Prediction using DNN

In [9]:
import numpy as np
import pandas as pd
import torch
from torch import nn

In [2]:
ds = pd.read_csv( "HousingData.csv" )
ds

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0.0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,NaN,22.4
502,0.04527,0.0,11.93,0.0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0.0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0.0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [3]:
ds.dropna(inplace=True)

In [5]:
def min_max_normalize(name):
  max_ = ds[name].max()
  min_ = ds[name].min()
  ds[name] = (ds[name] - min_) / ( max_ - min_ )

for col in ds.drop(["MEDV"], axis=1).columns:
    min_max_normalize(col)

In [10]:
from sklearn.model_selection import train_test_split

x = np.array(ds.drop(["MEDV"], axis=1))
y = np.array(ds["MEDV"])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=22)
y_train = np.expand_dims(y_train, axis=1)
y_test = np.expand_dims(y_test, axis=1)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

x_train = torch.Tensor(x_train)
y_train = torch.Tensor(y_train)
x_test = torch.Tensor(x_test)
y_test = torch.Tensor(y_test)

loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=16, shuffle=True)

(275, 13)
(119, 13)
(275, 1)
(119, 1)


In [13]:
model = nn.Sequential(
    nn.Linear(13, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

loss = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
def train_step(batch_x, batch_y):
  model.train()
  preds = model.forward(batch_x)
  batch_loss = loss(preds, batch_y)
  batch_loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  return batch_loss.item()

def train_epoch():
  epoch_loss = 0.0
  for batch_x, batch_y in loader:
    epoch_loss += train_step(batch_x, batch_y)
  return epoch_loss / len(loader)

def val_epoch():
  model.eval()
  preds = model.forward(x_test)
  val_loss = loss(preds, y_test)
  return val_loss.item()

for e in range(200):
  train_loss = train_epoch()
  val_loss = val_epoch()
  print(train_loss, val_loss)

22.244213316175674 21.313627243041992
22.259291542900932 21.0329647064209
21.74653392367893 20.332429885864258
20.787197960747612 18.653934478759766
17.771556880739 15.168377876281738
13.710445271597969 10.559231758117676
10.64849985970391 8.341017723083496
9.945645650227865 8.031726837158203
8.856500996483696 7.388885021209717
8.54112074110243 6.907404899597168
7.739466773139106 6.466545581817627
7.364378637737698 6.169426918029785
7.309217108620538 5.701613426208496
6.271172708935207 5.460537433624268
6.316641039318508 5.281322479248047
5.807841764556037 4.997469902038574
5.455283906724718 4.942163944244385
5.15381207731035 4.765317916870117
5.087776184082031 4.681435585021973
4.773518012629615 4.65125846862793
4.75196107228597 4.597661018371582
4.783658954832289 4.580767631530762
4.703128655751546 4.539431571960449
4.9456343915727405 4.537254333496094
4.619437760776943 4.5117387771606445
4.571794284714593 4.4926371574401855
4.409434874852498 4.4465179443359375
4.450059294700623 4.37